<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [1]:
%pip install torchinfo -qq
%pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
%pip install starfile -qq
%pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     / 10.7 MB 10.1 MB/s 0:00:01
  Preparing metadata (setup.py) ... done


> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [2]:
%pip install pycuda==2024.1
%pip install "numpy<2.0"
%pip install mrcfile -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 14.2 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2024.1-cp312-cp312-linux_x86_64.whl size=658547 sha256=767d83147c175704e9bea9408f2c9703a2c37fa2c09d322d518247142dc7a8f0
  Stored in directory: /root/.cache/pip/wheels/16/8d/d0/cf29b67ddc32f77bc3344b6b2b38dd9fe8595c726c33eabc03
Successfully built pycuda
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.9 MB/s eta 0:00:00


## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [1]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_default_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_default_output/results/10017_test/unet_eb5_dice_CDCRF" # @param {type:"string"}

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
IMAGE_DIR = "/content/image_dir"

In [7]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 270 (delta 141), reused 42 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 32.01 MiB | 12.07 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [3]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

> #### ⚠ Notice
>
> You need to restart the kernel after the compilation step. Additionally, if your GPU architecture differs, you may need to modify the file at `/content/CryoParticleSegment/Modeling/CRF_main/setup.py`. (We attempt to detect the GPU configuration automatically, so this step is usually not necessary.)
> Furthermore, depending on the number of classes and other parameters, you may encounter an error that requires adjusting the file at `/content/CryoParticleSegment/Modeling/CRF_main/src/PermutohedralFiltering/source/gpu/LatticeFilter.cu`:
>
> 1. Note the `pd` and `vd` values from the error message.
> 2. Add an `else if (pd == ? && vd == ?)` block.
> 3. Within that block, insert the following line, replacing `pd` and `vd` with the specific values:
>
>    ```cpp
>    latticeFilterGPU<pd, vd>(output_tensor, input_tensor, positions, num_super_pixels, backward);
>    ```
> 4. Recompile using `setup.py`.


In [9]:
#%git clone https://github.com/netw0rkf10w/CRF.git
%cd CryoParticleSegment/Modeling/CRF_main
!python setup.py clean --all
!rm -rf build/
!python setup.py build_ext --inplace --force
!python setup.py install

/content/CryoParticleSegment/Modeling/CRF_main
running clean
'build/lib.linux-x86_64-cpython-312' does not exist -- can't clean it
'build/bdist.linux-x86_64' does not exist -- can't clean it
'build/scripts-3.12' does not exist -- can't clean it
running build_ext
W1014 17:51:26.158000 5448 torch/utils/cpp_extension.py:615] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W1014 17:51:26.226000 5448 torch/utils/cpp_extension.py:507] The detected CUDA version (12.5) has a minor version mismatch with the version that was used to compile PyTorch (12.6). Most likely this shouldn't be a problem.
W1014 17:51:26.226000 5448 torch/utils/cpp_extension.py:517] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.5
building 'Permutohedral' extension
creating build/temp.linux-x86_64-cpython-312/src/PermutohedralFiltering/source/cpu
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEB

In [4]:
%cd /content/

/content


### ✅ Packages Handling

In [5]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [6]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [7]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [9]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = True # @param {type:"boolean"}


In [10]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [11]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [13]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
train_dataset =  MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset , batch_size=None, shuffle=False, pin_memory=True)

val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True)

full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True)

In [14]:
shape = None
for i1, i2, i3, i4 in val_loader: #test loader and reconstruct
    shape = i4.shape
    break

In [15]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [16]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:27<00:00,  3.04it/s]


In [18]:
processed_micrographs.shape

(84, 3840, 3840)

In [19]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [20]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [21]:
del model
torch.cuda.empty_cache()

In [22]:
model = model_post

In [23]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/TpzD_default_output/results/10017_test/unet_eb5_dice_CDCRF'

In [24]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()
        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [25]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [26]:
!cp {RESULT_DIR}/label_images.npy .

In [27]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [28]:
label_images.shape

(84, 3840, 3840)

In [29]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [30]:
!cp {RESULT_DIR}/best_config.txt .

In [31]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

### Actual seg

In [32]:
radius = RADIUS

In [33]:
from skimage import filters, segmentation, morphology
import skimage as ski
from skimage import img_as_float
from center_finding import normalize, min_rect_circle, eliminate_near
import cv2

In [34]:
from skimage import img_as_float
from skimage.filters import threshold_local

cv_list = []
for img in label_images:
    output_image = img_as_float(img)
    block_size = int(round(radius * 1.6)) | 1  # Ensure it's an odd number
    local_thresh = filters.threshold_local(output_image, block_size, method='gaussian', offset=0)
    image_seg = output_image > local_thresh
    thresh1 = ski.util.img_as_ubyte(image_seg)

    contr_min = radius*cv_config[1]
    kernel_size = int(radius / cv_config[0])  # Example ratio
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    thresh1 = cv2.erode(thresh1, kernel, iterations=1)

    contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_array = [c for c in contours]

    # Filter out small/large region and find bounding box with center
    c_ = np.array([cv2.contourArea(contour) for contour in contours])
    aa = (c_>contr_min) & (c_<500000)
    aa = aa.tolist()
    c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
    c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
    c_list = [x for x in c_list if x is not None]
    cv_list.append(c_list)
    print(len(c_full_list), len(c_list))

557 486
529 455
502 396
383 362
394 370
485 410
349 348
430 379
544 473
592 522
534 470
507 454
530 473
473 450
601 519
558 498
508 418
514 464
523 488
584 530
563 487
532 448
544 471
558 442
495 394
475 373
485 388
549 455
495 391
544 488
541 458
511 408
534 449
546 466
523 430
551 469
497 416
550 482
557 510
535 463
436 383
576 494
563 506
388 365
351 339
353 324
372 362
579 548
529 446
481 401
534 476
544 486
493 441
497 459
490 437
392 342
483 408
579 493
441 357
462 359
461 444
452 408
360 329
368 336
373 351
359 344
412 386
429 382
374 320
404 340
399 346
340 333
422 374
458 423
434 407
481 430
404 376
431 377
403 371
420 404
418 362
413 353
412 356
456 389


In [35]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in cv_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [36]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [37]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in cv_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [38]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,3526,3952,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,429,3951,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1510,3952,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,247,3948,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,3714,3944,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
35215,2363,186,00384@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35216,521,149,00385@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35217,315,147,00386@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35218,2857,154,00387@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [39]:
import starfile
starfile.write(df, f'{RESULT_DIR}/cv_particles.star')

### TrackPy

In [40]:
cv_list

[[(3398, 3824),
  (301, 3823),
  (1382, 3824),
  (119, 3820),
  (3586, 3816),
  (626, 3814),
  (2412, 3818),
  (3040, 3809),
  (1810, 3792),
  (1152, 3812),
  (2827, 3788),
  (1782, 3766),
  (2700, 3791),
  (1564, 3780),
  (3694, 3768),
  (803, 3784),
  (1698, 3764),
  (1296, 3731),
  (977, 3734),
  (3449, 3689),
  (3409, 3687),
  (469, 3684),
  (33, 3677),
  (1788, 3683),
  (365, 3672),
  (1137, 3652),
  (1531, 3659),
  (2766, 3658),
  (1436, 3666),
  (3054, 3650),
  (1953, 3625),
  (1020, 3625),
  (816, 3651),
  (2330, 3627),
  (996, 3564),
  (3098, 3555),
  (756, 3563),
  (3229, 3551),
  (1127, 3539),
  (41, 3530),
  (907, 3501),
  (676, 3516),
  (1585, 3479),
  (2366, 3487),
  (2997, 3497),
  (2278, 3453),
  (135, 3484),
  (1567, 3463),
  (2777, 3459),
  (602, 3468),
  (3781, 3461),
  (1776, 3457),
  (3386, 3446),
  (763, 3429),
  (1478, 3435),
  (2261, 3412),
  (2528, 3448),
  (1000, 3414),
  (16, 3408),
  (2931, 3403),
  (2644, 3412),
  (516, 3410),
  (1930, 3398),
  (885, 3369),

In [41]:
tp_config

(0.25, 0.4)

In [42]:
import trackpy as tp

In [43]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = round(TrackParticleSize*tp_config[1])
scale = tp_config[0]  # Scale factor (0.5 means reducing the size to half)

In [45]:
tp_list = []
for img in label_images:
    output_image = img
    small_image = cv2.resize(output_image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    # Adjust parameters based on the scale
    small_sep = int(sep * scale)
    small_diameter = int(TrackParticleSize * scale)
    if small_diameter % 2 == 0:
        small_diameter += 1
    coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
    #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
    coorTrack['x'] *= (1/scale)
    coorTrack['y'] *= (1/scale)
    coords = coorTrack[['x', 'y']].values
    tp_list.append(coords)
    print(len(coords))

603
583
625
365
387
551
313
450
593
620
569
485
561
464
645
571
556
540
497
588
590
638
606
683
593
599
604
636
617
559
594
634
591
597
593
613
560
603
543
610
474
626
572
367
332
364
344
546
596
559
566
560
505
503
517
428
569
643
627
605
431
471
365
365
355
338
406
456
423
460
442
307
460
435
409
491
395
466
406
398
449
456
439
466


In [46]:
tp_list

[array([[ 709.31572494,   93.72562263],
        [1013.82875074,   64.7335556 ],
        [3214.5845698 ,   49.63688258],
        ...,
        [2031.52043885, 3749.27395368],
        [3203.55834945, 3743.02543162],
        [1907.02775758, 3737.36421844]]),
 array([[2400.27177727,   93.71179578],
        [ 225.87005906,   79.75505205],
        [3679.48940228,   47.51306835],
        ...,
        [3600.86630873, 3784.80287368],
        [ 477.93980477, 3763.9070344 ],
        [2778.24170153, 3782.48211276]]),
 array([[ 282.2452636 ,   62.54877444],
        [3728.23503974,   43.84864562],
        [2401.91981362,  110.78713411],
        ...,
        [3550.71852693, 3793.12370214],
        [1962.1627589 , 3765.77410915],
        [3690.26562906, 3780.16588313]]),
 array([[2920.4917967 ,   63.69040074],
        [ 875.39158118,   51.30094259],
        [2249.26448011,   51.22002791],
        [3562.99443181,   59.75241158],
        [1259.49304925,   63.04016766],
        [1422.50932411,   87.441903

In [47]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in tp_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [48]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in tp_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [49]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,837.315725,221.725623,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1141.828751,192.733556,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,3342.584570,177.636883,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,1288.430135,233.311136,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,1466.816616,223.738010,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
43016,3754.404280,3819.852911,00461@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
43017,188.069955,3823.475202,00462@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
43018,1627.363095,3861.655478,00463@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
43019,1521.787563,3909.883205,00464@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [50]:
import starfile
starfile.write(df, f'{RESULT_DIR}/tp_particles.star')

### Nonmax

In [51]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [52]:
nms_config

(0.5, 0.6)

In [53]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more
pCan=nms_config[0] #the grid size smaller will generate more candidate
overlap=nms_config[1] #allow for overlap
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [54]:
nms_list = []
for idx, img in enumerate(label_images):
    # if idx == 6:
    #     break
    heatArr=pad_image(img)
    heatArr=reNorm(heatArr, nSep)
    heatArr=reLev(heatArr,level)
    gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

    # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
    gsize=psize*pCan
    heat=heatArr.astype(np.float32)
    gaus=gaus.astype(np.float32)
    score=np.zeros(heat.shape).astype(np.float32)
    [sizex,sizey]=heat.shape
    sizex = int(sizex)
    sizey = int(sizey)
    psize = int(psize)
    gsize = int(gsize)

    func = scoreGpu

    tx=16
    ty=16
    bx=(sizex-1)//tx+1
    by=(sizey-1)//ty+1
    print('get score gpu',tx, ty, bx, by, gsize, psize)


    func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
        block=(tx, ty, 1), grid=(int(bx), int(by)))

    # # Calculate necessary dimensions and convert all to int
    numx = int((sizex - 1) // gsize + 1)
    numy = int((sizey - 1) // gsize + 1)
    num = numx * numy
    tnum = num * 5

    # # Create the result array
    res = np.zeros(tnum).astype(np.float32)

    # # Block and grid dimensions
    bx = (numx - 1) // tx + 1
    by = (numy - 1) // ty + 1

    # Get the function from the module
    func = getMax

    # Call the function, ensure all parameters are the correct type
    func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
        block=(16, 16, 1), grid=(bx, by, 1))

    # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
    niter = np.int32(nIter)
    gsize = np.int32(gsize)
    sizex = np.int32(sizex)
    sizey = np.int32(sizey)
    numx = np.int32(numx)
    tnum = np.int32(num)

    print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
    func = getMax3
    # Ensuring the correct parameter order and type for the kernel invocation
    func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
            block=(16, 16, 1), grid=(bx, by, 1))

    canList=reshape(res,num)

    print('Number of Particles before:', len(canList))
    if(len(canList)>pnum):
        canList=canList[:pnum]


    canList=cleanCanList(canList,overlap,psize)
    #canList=reCan(canList,ratio)
    print('Number of Particles:', len(canList))
    nms_list.append([(r[1],r[0]) for r in canList])

get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3467
Number of Particles: 638
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3466
Number of Particles: 607
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3474
Number of Particles: 666
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3248
Number of Particles: 444
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3268
Number of Particles: 444
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3416
Number of Particles: 604
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3

In [55]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in nms_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [56]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [57]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,2300.8,3760.8,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,2799.2,3098.2,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1137.6,1786.4,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,3043.0,430.0,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,432.0,1867.0,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
47595,2395.0,1024.0,00541@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
47596,2815.0,2304.0,00542@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
47597,3455.0,1867.0,00543@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
47598,3071.0,1983.0,00544@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [58]:
starfile.write(df, f'{RESULT_DIR}/nms_particles.star')